> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


## 实验九：GEMM算子的优化与验证


建议学时：4学时


## 实验任务


1.任务描述


本实验在GEMM基础版基础上实现优化版。优化版采用M/N/K分块、B矩阵转置布局和K维向量化归约，减少不连续访存和标量循环开销。


2.学习目标


完成本实验后，学生应能够：


• 理解GEMM优化中分块、数据布局和向量化归约之间的关系。


• 掌握B_T连续布局对K维向量读取的意义。


• 能够使用DataCopy、Mul和ReduceSum实现局部内积。


• 能够分析优化版相对基础版的收益以及与原生GEMM的差距。


## 任务准备


1.优化前的瓶颈与策略总览


GEMM基础版直接按照矩阵乘法定义实现三重循环，便于理解语义，但在模型级替换中会暴露出访存不连续、累加粒度小和任务划分粗等问题。优化版围绕分块、连续布局和向量归约展开，在保持矩阵乘法语义不变的前提下重组计算流程。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">优化前瓶颈</th>
<th style="text-align:left;">影响</th>
<th style="text-align:left;">优化策略</th>
<th style="text-align:left;">代码应用</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">B列方向访问不连续</td>
<td style="text-align:left;">K维累加时访存跨度较大</td>
<td style="text-align:left;">B_T连续布局</td>
<td style="text-align:left;">注册侧生成b.transpose(0,1).contiguous()</td>
</tr>
<tr>
<td style="text-align:left;">输出任务划分较粗</td>
<td style="text-align:left;">只按行划分不利于更细粒度并行</td>
<td style="text-align:left;">M/N分块</td>
<td style="text-align:left;">使用tileM/tileN/totalTiles描述输出块</td>
</tr>
<tr>
<td style="text-align:left;">K维标量累加</td>
<td style="text-align:left;">每个输出元素逐项乘加</td>
<td style="text-align:left;">K分块与向量归约</td>
<td style="text-align:left;">使用Mul + ReduceSum计算局部内积</td>
</tr>
<tr>
<td style="text-align:left;">全局内存访问频繁</td>
<td style="text-align:left;">数据复用不足</td>
<td style="text-align:left;">UB片段搬运</td>
<td style="text-align:left;">A片段和B_T片段搬入LocalTensor后计算</td>
</tr>
</tbody></table>


GEMM优化不是单纯把循环改写成向量接口。矩阵乘法的关键是让数据访问连续、让一段数据在片上缓冲区中被充分使用，并尽量减少重复从全局内存读取。优化版仍计算标准矩阵乘法。


本实验中的GEMM优化围绕三个策略展开：分块、连续布局和向量化归约。分块的原理是把大的输出矩阵C[M,N]拆成多个较小的输出块，每个算核负责若干个块。这样可以让任务分配更细，也能为片上缓冲区中的局部计算创造条件。当前实现使用tileM、tileN和tileK分别描述M方向、N方向和K方向的分块粒度，其中M/N分块决定输出块形状，K分块决定每次局部内积处理多少个累加元素。


连续布局的原理是让向量读取尽量面对连续地址。原始矩阵B[K,N]按行主序存储时，计算某个输出列需要沿B的列方向读取，地址跨度为N，不利于连续搬运。优化版在注册侧先把B转置为B_T[N,K]，这样kernel计算C[row,col]时，可以从B_T[col,kStart]开始读取一段连续K维数据。这个布局变化没有改变数学公式，只是把device侧访问方式改得更适合向量化。


向量化归约的核心原理，是将 K 维度内积运算中逐次标量相乘累加的过程，改写为批量向量乘法搭配分段求和。基础实现方案针对 K 维度的每一个取值依次完成标量的乘法与累加；优化实现则在点积分块函数内，先将矩阵 A、转置后矩阵 B 的对应分块数据搬运至统一缓存空间，通过向量运算完成批量相乘，再借助归约求和得到该段 K 维度对应的局部累加结果。外层循环不断累加各个 K 分块算出的局部结果，最终得到目标输出矩阵的元素。经过优化，矩阵通用乘法的计算逻辑由原本逐项串行累加，转变为分块数据搬运、批量向量运算、分段归约求和的完整执行流程。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">输出块：把C[M,N]切成tile，分配给多个算核 B_T布局：把B[K,N]转为B_T[N,K]，让K维片段连续 向量归约：对一段K维数据执行Mul+ReduceSum，得到局部内积</th>
</tr>
</thead>
</table>


2.优化前后对比


下表对比GEMM基础版和优化版在任务划分、数据布局、K维计算和模型级结果上的差异。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">对比项</th>
<th style="text-align:left;">基础版</th>
<th style="text-align:left;">优化版</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">任务划分</td>
<td style="text-align:left;">按输出行划分</td>
<td style="text-align:left;">按输出矩阵tile划分</td>
</tr>
<tr>
<td style="text-align:left;">B矩阵布局</td>
<td style="text-align:left;">读取原始B[K,N]</td>
<td style="text-align:left;">注册侧生成B_T[N,K]</td>
</tr>
<tr>
<td style="text-align:left;">K维计算</td>
<td style="text-align:left;">标量循环累加</td>
<td style="text-align:left;">分块后执行向量乘法和归约</td>
</tr>
<tr>
<td style="text-align:left;">片上缓冲</td>
<td style="text-align:left;">基本不使用UB复用</td>
<td style="text-align:left;">使用队列和workBuf组织局部内积</td>
</tr>
<tr>
<td style="text-align:left;">模型测试结果</td>
<td style="text-align:left;">custom路径约15254.788 ms</td>
<td style="text-align:left;">custom路径约1292.354 ms</td>
</tr>
</tbody></table>


3.算子定义与接口约定


优化版GEMM仍对外提供torch.ops.gemm_custom.gemm(a, b)接口。与基础版相比，接口内部会额外生成B_T连续布局，再把该布局传入device侧kernel。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">工程约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">输入</td>
<td style="text-align:left;">A[M,K]和B[K,N]，注册侧传入kernel前生成B_T[N,K]</td>
</tr>
<tr>
<td style="text-align:left;">输出</td>
<td style="text-align:left;">C[M,N]</td>
</tr>
<tr>
<td style="text-align:left;">优化策略</td>
<td style="text-align:left;">M/N/K分块、B_T连续布局、K维向量归约</td>
</tr>
<tr>
<td style="text-align:left;">核心参数</td>
<td style="text-align:left;">tileM=8、tileN=8、tileK=128</td>
</tr>
<tr>
<td style="text-align:left;">调用方式</td>
<td style="text-align:left;">torch.ops.gemm_custom.gemm(a, b)</td>
</tr>
</tbody></table>


4.实验环境准备


本实验在Ascend NPU云服务器上完成，使用CANN工具链、AscendC和PyTorch NPU环境。基础版与优化版建议放在不同工程目录中，构建和测试也尽量在新的Python进程中执行，避免torch.library重复注册。


进入工程目录后，先加载CANN环境变量，再检查环境、构建工程并设置动态库搜索路径。CANN安装路径以云服务器实际配置为准。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">cd /home/user/GemmOptimizedExperiment source /usr/local/Ascend/ascend-toolkit/set_env.sh bash scripts/check_env.sh bash scripts/build.sh export LD_LIBRARY_PATH=$PWD/out/lib:$LD_LIBRARY_PATH</th>
</tr>
</thead>
</table>


## 任务实施


## 步骤一：明确优化思路


本步骤说明GEMM优化版的三项核心改造：输出矩阵分块、B矩阵转置后连续读取、K维向量归约。它们共同决定了优化版如何从“按元素累加”转向“按tile处理”，也是整份优化手册的主线。


基础版GEMM沿K维逐项访问B的列，内存访问不连续，且每次累加都是标量操作。优化版从三个方向改造：第一，按M/N把输出矩阵切成小块；第二，在注册侧把B转置为B_T[N,K]，让算子内核读取B的K维片段时连续；第三，把K维的一段数据搬入片上缓冲区，使用向量乘法和归约得到局部累加值。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">基础版：按行分工 -&gt; B列方向非连续访问 -&gt; 标量累加 优化版：按输出块分工 -&gt; B_T连续访问 -&gt; K维向量乘法与归约</th>
</tr>
</thead>
</table>


## 步骤二：设计M/N/K分块参数


本步骤把输出矩阵拆成M方向和N方向的小块，并为每个小块定义统一的tile大小。tileM和tileN决定输出块的形状，tileK决定每次局部内积所处理的累加长度。它的功能是把大矩阵拆成适合分配和计算的单位。


优化版tiling保存矩阵规模、算核数量、M/N/K三个方向的分块大小以及总分块数量。totalTiles表示输出矩阵被切成多少个小块，算子内核通过tileId把这些小块分配给不同算核。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">#pragma pack(push, 1) struct GemmOptimizedTiling { uint32_t m = 0; uint32_t n = 0; uint32_t k = 0; uint32_t coreNum = 1; uint32_t tileM = 8; uint32_t tileN = 8; uint32_t tileK = 128; uint32_t totalTiles = 0; }; #pragma pack(pop)</th>
</tr>
</thead>
</table>


注册侧根据矩阵规模计算分块数量。当前默认tileM=8、tileN=8、tileK=128，这些参数便于展示片上缓冲区搬运和向量归约流程。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">GemmOptimizedTiling BuildTiling(uint32_t m, uint32_t n, uint32_t k, uint32_t blockDim) { GemmOptimizedTiling t {}; t.m = m; t.n = n; t.k = k; t.coreNum = std::max(1U, std::min(blockDim, 32U)); t.tileM = kTileM; t.tileN = kTileN; t.tileK = kTileK; const uint32_t mTiles = (t.m + t.tileM - 1U) / t.tileM; const uint32_t nTiles = (t.n + t.tileN - 1U) / t.tileN; t.totalTiles = mTiles * nTiles; return t; }</th>
</tr>
</thead>
</table>


## 步骤三：在注册侧生成连续的B_T布局


本步骤在host侧把B[K,N]转成B_T[N,K]。这样做以后，kernel读取某个输出列对应的K维片段时，就可以按连续地址访问数据。它是整个优化版能否成立的关键，因为数据布局直接决定后面的向量化是否顺畅。


原始B[K,N]按列读取时步长为N，不适合直接向量化。注册侧先执行转置并调用contiguous()，把它变成B_T[N,K]。算子内核随后读取bTransGm_[col*k+kStart]，就能得到连续的K维片段。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">auto aContig = a.contiguous(); auto bTrans = b.transpose(0, 1).contiguous(); auto c = torch::empty({a.size(0), b.size(1)}, a.options()); const uint32_t launchRet = ACLRT_LAUNCH_KERNEL(gemm_optimized_kernel)( realBlockDim, stream, static_cast&lt;uint8_t *&gt;(aContig.data_ptr()), static_cast&lt;uint8_t *&gt;(bTrans.data_ptr()), static_cast&lt;uint8_t *&gt;(c.data_ptr()), static_cast&lt;uint8_t *&gt;(workspaceDev), static_cast&lt;uint8_t *&gt;(tilingDev));</th>
</tr>
</thead>
</table>


## 步骤四：初始化片上队列和工作缓冲区


该步骤用于为通用矩阵乘法的分块内积运算完成片上存储资源的规划配置。矩阵 A 的分块数据、转置后矩阵 B 的分块数据经由输入队列搬运至本地缓存；向量相乘结果、归约计算所需临时存储空间存放于工作缓存区，最终输出的分块数据则借助输出队列临时缓存。整套资源规划保证每一个计算分块的全部运算流程都在芯片片上存储中完成。


优化方案为矩阵 A 分块、转置 B 分块、输出分块分别配置专属数据队列，同时使用工作缓存承载乘积向量、归约临时存储区域以及归约最终结果。该缓存的容量必须能够完整覆盖 K 维度分块长度，若容量不足，当输入矩阵的 K 维度尺寸较大时，归约运算环节极易出现运行错误。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">const uint32_t tileBytes = tileK_ * sizeof(float); pipe_.InitBuffer(aQueue_, 1, tileBytes); pipe_.InitBuffer(bQueue_, 1, tileBytes); pipe_.InitBuffer(cQueue_, 1, tileN_ * sizeof(float)); pipe_.InitBuffer(workBuf_, (2U * tileK_ + 8U) * sizeof(float));</th>
</tr>
</thead>
</table>


## 步骤五：实现K维向量乘法与归约


本步骤将单个输出元素沿 K 维度的逐项累加运算，重构为向量乘法搭配归约求和的运算形式。分块内积计算函数DotTile( )会先将矩阵 A、转置矩阵 B 对应的分块数据搬运至片上缓存，通过逐元素相乘得到完整乘积向量，再借助归约求和运算，算出当前 K 维度分段对应的累加结果，直观实现了由标量逐项累加向向量批量归约计算的转型。


分块内积函数的核心作用，是求解单个输出元素在某一段 K 维度区间内的局部内积。运算流程为先载入矩阵 A 与转置矩阵 B 的对应分段数据，完成逐元素相乘得到乘积向量，再经归约求和得到该段 K 维度的局部累加值。外层循环依次汇总所有 K 维度分段的局部累加结果，最终算出输出矩阵对应行列位置的最终数值。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;"><strong>aicore</strong> inline float DotTile( uint32_t row, uint32_t col, uint32_t kStart, uint32_t curK) { LocalTensor<float> aLocal = aQueue_.AllocTensor<float>(); LocalTensor<float> bLocal = bQueue_.AllocTensor<float>(); DataCopy(aLocal, aGm_[row * k_ + kStart], curK); DataCopy(bLocal, bTransGm_[col * k_ + kStart], curK); aQueue_.EnQue(aLocal); bQueue_.EnQue(bLocal); aLocal = aQueue_.DeQue<float>(); bLocal = bQueue_.DeQue<float>(); LocalTensor<float> workLocal = workBuf_.Get<float>(); LocalTensor<float> prodLocal = workLocal; LocalTensor<float> reduceTmpLocal = workLocal[tileK_]; LocalTensor<float> reduceResultLocal = workLocal[2U * tileK_]; Mul(prodLocal, aLocal, bLocal, curK); PipeBarrier<PIPE_V>(); ReduceSum<float>(reduceResultLocal, prodLocal, reduceTmpLocal, static_cast<int32_t>(curK)); PipeBarrier<PIPE_V>(); const float partial = reduceResultLocal.GetValue(0); aQueue_.FreeTensor(aLocal); bQueue_.FreeTensor(bLocal); return partial; }</th>
</tr>
</thead>
</table>


## 步骤六：按输出块写回结果


本步骤将分片求得的局部内积结果规整排布，写回至最终输出矩阵。分块处理流程会先定位当前计算分块在输出矩阵中对应的行分块、列分块位置，再逐行将连续的计算片段写入输出矩阵 C。该操作能够保障输出数据和目标矩阵尺寸完全匹配，完成全部运算结果的落地存储。


外层处理逻辑会将分块编号映射为输出矩阵 M 维度、N 维度对应的分块序号。在每一个分块范围内逐行计算该行实际需要输出的元素数量，借助一次批量数据拷贝操作，把该行位于当前分块内的连续计算结果整体回写。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">for (uint32_t tileId = coreId; tileId &lt; totalTiles_; tileId += coreNum_) { const uint32_t mTileId = tileId / nTiles; const uint32_t nTileId = tileId % nTiles; const uint32_t mStart = mTileId * tileM_; const uint32_t nStart = nTileId * tileN_; const uint32_t curM = Min(tileM_, m_ - mStart); const uint32_t curN = Min(tileN_, n_ - nStart); ProcessOneCTile(mStart, nStart, curM, curN); }</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">python3 tests/test_torch_op.py --m 128 --k 1024 --n 512 --atol 1e-3 --rtol 1e-3 python3 tests/test_qwen_linear.py --batch 1 --seq 128 --hidden 1024 --out 512 python3 tests/compare_qwen_native.py \ --model /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Models/Qwen2.5-0.5B \ --repeat 3 \ --attn-implementation eager</th>
</tr>
</thead>
</table>


# 测试与验收


## 一、单算子正确性测试


该测试调用当前实验的PyTorch注册算子，并与同一数学语义的参考实现比较。终端输出全部为PASS（或ALL PASS）且进程返回码为0，表示正确性测试通过。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/GemmOptimizedExperiment
bash scripts/build.sh
python3 tests/test_torch_op.py


## 二、单算子执行时间测试


该指令先预热，再重复启动单个算子，并使用ACL Event统计设备侧执行时间。记录输出中的mean、median、min和max；该结果不包含Python参考计算、输入生成、结果比对及首次主机到设备的数据传输。基础版与优化版比较时，应使用相同输入形状、预热次数、重复次数和计算核心数。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/GemmOptimizedExperiment
export LD_LIBRARY_PATH="$PWD/out/lib:${LD_LIBRARY_PATH:-}"
./out/bin/gemm_optimized_standalone --m 128 --k 1024 --n 512 --block-dim 16 --warmup 10 --repeat 20 --rounds 5


## 任务拓展


完成基础实验后，可以继续围绕以下方向拓展：


• 在相同输入规模下对比基础版、优化版和原生算子的耗时。


• 调整任务划分和分块参数，观察正确性、吞吐和尾块处理是否变化。


• 将单算子测试、独立直调测试和模型替换测试的结果放在一起分析，区分算子内核内部耗时与端到端调度开销。


• 进一步尝试双缓冲、异步搬运、多级流水、FP16/BF16支持或更贴近硬件矩阵单元的实现。


## 实验总结


本次实验完整呈现了通用矩阵乘法由基础三层循环朴素写法，逐步改造为分块向量化高性能实现的全流程。整个实验的核心学习重点围绕三大关键问题展开：输出矩阵的分块划分规则、矩阵 B 执行转置操作的底层原因、K 维度内积借助向量乘法与归约运算重构的实现逻辑。


优化版本的矩阵乘法进一步体现：矩阵乘法的性能优化不只是运算执行方式的调整，本质更是内存数据排布规则与硬件任务拆分逻辑的系统性重构。依托输出矩阵分块、矩阵 B 转置、K 维度向量归约三项核心优化手段，能够清晰展现朴素串行代码走向结构化硬件适配优化的完整思路，充分理解面向硬件的算子优化方法。
